# EuroSAT 数据集预处理与划分 (V2)

**目标**: 
该脚本将扫描 `EuroSAT_RGB` 文件夹，并生成用于训练、验证和测试的元数据文件。

**核心功能**:
1.  在 `EuroSAT_RGB` 的同级目录下创建一个名为 `split_info` 的新文件夹。
2.  在该文件夹内生成以下文件：
    - `label_map.json`: 类别名称到数字索引的映射。
    - `train.csv`: 训练集 (70%) 索引。
    - `valid.csv`: 验证集 (15%) 索引。
    - `test.csv`: 测试集 (15%) 索引。

In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import json

## 1. 配置路径

请根据您的项目结构，正确设置 `project_path`。

In [5]:
# --- 关键配置区 ---
# 这是包含 EuroSAT_RGB 和未来 split_info 文件夹的父目录
# project_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/"
project_path = "F:/vscode project/IMA_PW3/data"

# 自动构建图片数据路径和元数据输出路径
image_data_path = os.path.join(project_path, "EuroSAT_RGB/").replace('\\', '/')
output_path = os.path.join(project_path, "split_info/").replace('\\', '/')

# 确保输出目录存在
os.makedirs(output_path, exist_ok=True)

print(f"图片数据将从这里读取: {image_data_path}")
print(f"生成的CSV和JSON文件将保存在: {output_path}")

图片数据将从这里读取: F:/vscode project/IMA_PW3/data/EuroSAT_RGB/
生成的CSV和JSON文件将保存在: F:/vscode project/IMA_PW3/data/split_info/


## 2. 扫描、划分并保存数据信息

In [6]:
def create_dataset_splits(image_dir, output_dir, train_ratio=0.7, valid_ratio=0.15, test_ratio=0.15):
    """
    根据指定的路径和比例，生成数据集划分文件。
    """
    if not os.path.isdir(image_dir):
        print(f"错误: 图片目录 '{image_dir}' 不存在。")
        return

    print("--- 开始EuroSAT数据集预处理 ---")

    # --- 步骤 1: 扫描文件夹，创建类别到数字标签的映射 ---
    class_names = [d for d in os.listdir(image_dir) if os.path.isdir(os.path.join(image_dir, d))]
    class_names.sort()
    label_to_index = {name: i for i, name in enumerate(class_names)}
    print(f"发现 {len(class_names)} 个类别: {class_names}")

    # --- 步骤 2: 收集所有图片的相对路径和标签 ---
    image_paths = []
    labels = []
    print("正在收集所有图片的路径和标签...")
    for class_name, label_idx in label_to_index.items():
        class_dir = os.path.join(image_dir, class_name)
        for image_name in os.listdir(class_dir):
            relative_path = os.path.join(class_name, image_name).replace('\\', '/')
            image_paths.append(relative_path)
            labels.append(label_idx)

    df_all = pd.DataFrame({'ImagePath': image_paths, 'Label': labels})
    print(f"共找到 {len(df_all)} 张图片。")

    # --- 步骤 3: 按比例进行分层抽样划分 ---
    print(f"正在将数据集划分为: {train_ratio:.0%} 训练集, {valid_ratio:.0%} 验证集, {test_ratio:.0%} 测试集...")
    train_val_df, test_df = train_test_split(
        df_all, test_size=test_ratio, random_state=42, stratify=df_all['Label']
    )
    val_split_ratio = valid_ratio / (train_ratio + valid_ratio)
    train_df, valid_df = train_test_split(
        train_val_df, test_size=val_split_ratio, random_state=42, stratify=train_val_df['Label']
    )

    # --- 步骤 4: 打印划分结果总结 ---
    print("\n--- 数据集划分总结 ---")
    print(f"训练集:   {len(train_df)} 张图片 ({len(train_df)/len(df_all):.2%})")
    print(f"验证集: {len(valid_df)} 张图片 ({len(valid_df)/len(df_all):.2%})")
    print(f"测试集:       {len(test_df)} 张图片 ({len(test_df)/len(df_all):.2%})")

    # --- 步骤 5: 将结果保存到指定的输出目录 ---
    train_csv_path = os.path.join(output_dir, 'train.csv')
    valid_csv_path = os.path.join(output_dir, 'valid.csv')
    test_csv_path = os.path.join(output_dir, 'test.csv')
    label_map_path = os.path.join(output_dir, 'label_map.json')

    print(f"\n正在保存元数据文件到 '{output_dir}' 目录...")
    train_df.to_csv(train_csv_path, index=False)
    valid_df.to_csv(valid_csv_path, index=False)
    test_df.to_csv(test_csv_path, index=False)
    with open(label_map_path, 'w') as f:
        json.dump(label_to_index, f, indent=4)
        
    print("\n--- 预处理完成! ---")

# --- 运行主函数 ---
create_dataset_splits(image_data_path, output_path)

--- 开始EuroSAT数据集预处理 ---
发现 10 个类别: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
正在收集所有图片的路径和标签...
共找到 27000 张图片。
正在将数据集划分为: 70% 训练集, 15% 验证集, 15% 测试集...

--- 数据集划分总结 ---
训练集:   18899 张图片 (70.00%)
验证集: 4051 张图片 (15.00%)
测试集:       4050 张图片 (15.00%)

正在保存元数据文件到 'F:/vscode project/IMA_PW3/data/split_info/' 目录...

--- 预处理完成! ---


## 5. 如何修改主训练代码 (`EuroSat_ResNeXt_Classifier.ipynb`)

为了让您的主训练代码能够找到这些新位置的文件，您需要做两个简单的修改：

**A. 修改路径变量**

在主代码中找到定义路径的单元格（在您的文件中是单元格 6 和 8），将它们修改为指向新的 `split_info` 文件夹，像这样：

```python
# 图片所在的文件夹
data_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/EuroSAT_RGB/"

# CSV和JSON文件所在的文件夹
metadata_path = "F:/vscode project/IMA_PW3/EuroSat_classification-main_data_wx/split_info/"

# 加载标签映射文件
f = open(os.path.join(metadata_path, "label_map.json"), "r")
label_to_index = json.load(f)
```

**B. 修改 `EurosatDataset` 的调用**

在主代码中找到创建 `train_dataset`, `valid_dataset`, `test_dataset` 的单元格（单元格 12），将它们修改为从 `metadata_path` 中读取CSV文件。

**原来的代码:**
```python
train_dataset = EurosatDataset(_type='train', transform=transformToTensor, data_path=data_path)
```

**修改后的代码 (您需要同步修改 `EuroSAT.py` 文件中的 `EurosatDataset` 类来接收 `csv_path`):**
```python
# 假设 EurosatDataset 类被修改为接收 csv_path
train_dataset = EurosatDataset(
    csv_path=os.path.join(metadata_path, 'train.csv'), 
    transform=transformToTensor, 
    data_path=data_path
)
valid_dataset = EurosatDataset(
    csv_path=os.path.join(metadata_path, 'valid.csv'), 
    transform=transformToTensor, 
    data_path=data_path
)
test_dataset = EurosatDataset(
    csv_path=os.path.join(metadata_path, 'test.csv'), 
    transform=transformToTensor, 
    data_path=data_path
)
```

要实现这一点，您需要将您的 `EuroSAT.py` 文件中的 `EurosatDataset` 类修改成类似这样：

```python
class EurosatDataset(Dataset):
    def __init__(self, csv_path, data_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.data_path = data_path
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_rel_path = self.data.iloc[idx, 0]
        img_abs_path = os.path.join(self.data_path, img_rel_path)
        label = self.data.iloc[idx, 1]
        image = Image.open(img_abs_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label
```

这样就完成了所有必要的调整！